In [ ]:
# pip Install kubeflow SDK from main branch for testing
%pip install git+https://github.com/opendatahub-io/kubeflow-sdk.git@main

In [ ]:
import os
import urllib3
from kubernetes import client as k8s

# Suppress InsecureRequestWarning since we use verify_ssl = False
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

api_server = os.getenv("OPENSHIFT_API_URL")
token = os.getenv("NOTEBOOK_USER_TOKEN")
if not api_server or not token:
    raise RuntimeError("OPENSHIFT_API_URL and NOTEBOOK_USER_TOKEN environment variables are required")

PVC_NAME = os.getenv("SHARED_PVC_NAME", "shared")

configuration = k8s.Configuration()
configuration.host = api_server
configuration.verify_ssl = False
configuration.api_key = {"authorization": f"Bearer {token}"}
api_client = k8s.ApiClient(configuration)

PVC_MOUNT_PATH = "/opt/app-root/src"

In [ ]:
from kubeflow.trainer import TrainerClient
from kubeflow.common.types import KubernetesBackendConfig

backend_cfg = KubernetesBackendConfig(
    client_configuration=api_client.configuration,
)

client = TrainerClient(backend_cfg)
print(client)

In [ ]:
training_runtime_name = os.getenv("TRAINING_RUNTIME")
if not training_runtime_name:
    raise RuntimeError("TRAINING_RUNTIME environment variable is required")

th_runtime = None
for runtime in client.list_runtimes():
    if runtime.name == training_runtime_name:
        th_runtime = runtime
        print("Found runtime: " + str(th_runtime))
        break

if th_runtime is None:
    raise RuntimeError(f"Required runtime '{training_runtime_name}' not found")

In [ ]:
import time
from kubernetes import client as k8s_client

from kubeflow.trainer.rhai import TrainingHubAlgorithms, TrainingHubTrainer
from kubeflow.trainer.options.kubernetes import (
    PodTemplateOverrides,
    PodTemplateOverride,
    PodSpecOverride,
    ContainerOverride,
)

# K8s CoreV1 API for pod status inspection
core_v1 = k8s_client.CoreV1Api(api_client)

NAMESPACE = os.getenv("NOTEBOOK_NAMESPACE")
print(f"Using namespace: {NAMESPACE}")


def get_training_pod_status(namespace, job_name):
    """Find training pods for a TrainJob and return restart counts and statuses."""
    # Use the jobset label which matches the TrainJob name
    selector = f"jobset.sigs.k8s.io/jobset-name={job_name}"
    try:
        pods = core_v1.list_namespaced_pod(
            namespace=namespace,
            label_selector=selector,
        )
    except Exception as e:
        print(f"  Pod list error with selector '{selector}': {e}")
        return []

    if not pods.items:
        # Fallback: list all pods and find by name prefix
        try:
            all_pods = core_v1.list_namespaced_pod(namespace=namespace)
            pods_items = [p for p in all_pods.items if p.metadata.name.startswith(job_name)]
            if pods_items:
                pods.items = pods_items
        except Exception:
            return []

    results = []
    for pod in pods.items:
        for cs in (pod.status.container_statuses or []):
            state = "unknown"
            if cs.state.waiting:
                state = cs.state.waiting.reason or "Waiting"
            elif cs.state.running:
                state = "Running"
            elif cs.state.terminated:
                state = f"Terminated({cs.state.terminated.reason})"
            results.append({
                "pod": pod.metadata.name,
                "container": cs.name,
                "restart_count": cs.restart_count,
                "state": state,
            })
    return results


def run_failure_scenario(client, scenario_name, training_params, expected_error, runtime, pvc_name, algorithm=TrainingHubAlgorithms.SFT):
    """Submit a TrainJob expected to fail, verify the expected error in logs
    and that the container is crashing (restart count > 0)."""
    print(f"\n{'='*60}")
    print(f"Scenario: {scenario_name}")
    print(f"Expected error: {expected_error}")
    print(f"{'='*60}")

    job_name = None

    try:
        job_name = client.train(
            trainer=TrainingHubTrainer(
                algorithm=algorithm,
                func_args=training_params,
                resources_per_node={
                    "cpu": 4,
                    "memory": "16Gi",
                },
            ),
            options=[
                PodTemplateOverrides(
                    PodTemplateOverride(
                        target_jobs=["node"],
                        spec=PodSpecOverride(
                            volumes=[
                                {"name": "work", "persistentVolumeClaim": {"claimName": pvc_name}},
                            ],
                            containers=[
                                ContainerOverride(
                                    name="node",
                                    volume_mounts=[
                                        {"name": "work", "mountPath": "/opt/app-root/src", "readOnly": False},
                                    ],
                                )
                            ],
                        ),
                    )
                )
            ],
            runtime=runtime,
        )
        print(f"TrainJob created: {job_name}")

        # Wait for the job to start running
        try:
            client.wait_for_job_status(name=job_name, status={"Running", "Failed"}, timeout=300)
        except Exception as e:
            print(f"Wait for Running/Failed raised: {e}")

        # Poll until we see: (1) the expected error in logs, and (2) container has crashed
        error_found = False
        crash_confirmed = False
        deadline = time.time() + 300  # 5 minute timeout

        while time.time() < deadline:
            # Check logs for the expected error
            if not error_found:
                try:
                    log_lines = list(client.get_job_logs(job_name, follow=False))
                    log_text = "\n".join(str(line) for line in log_lines)
                    if expected_error in log_text:
                        error_found = True
                        print(f"Found expected error in logs: '{expected_error}'")
                except Exception as e:
                    print(f"Log fetch error (retrying): {e}")

            # Check pod restart count or crash state
            try:
                pod_statuses = get_training_pod_status(NAMESPACE, job_name)
                for ps in pod_statuses:
                    if not crash_confirmed:
                        if ps["restart_count"] > 0:
                            crash_confirmed = True
                            print(f"Container crash confirmed (restart_count={ps['restart_count']})")
                        elif ps["state"] in ("CrashLoopBackOff", "Error"):
                            crash_confirmed = True
                            print(f"Container crash confirmed (state={ps['state']})")
                        elif "Terminated" in ps["state"]:
                            crash_confirmed = True
                            print(f"Container crash confirmed (terminated)")
            except Exception:
                pass

            if error_found and crash_confirmed:
                break

            time.sleep(15)

        # Verify results
        assert error_found, f"Expected error '{expected_error}' not found in logs"
        assert crash_confirmed, (
            f"Container did not crash — final pod statuses: "
            f"{get_training_pod_status(NAMESPACE, job_name)}"
        )

        print(f"PASSED: {scenario_name}")
        result = True

    except Exception as e:
        print(f"FAILED: {scenario_name} - {e}")
        result = False

    finally:
        # Cleanup
        if job_name:
            try:
                client.delete_job(job_name)
                print(f"Deleted job: {job_name}")
            except Exception:
                pass

    return result

In [ ]:
# Scenario 1: Invalid Dataset Path
result_1 = run_failure_scenario(
    client=client,
    scenario_name="Invalid Dataset Path",
    training_params={
        "model_path": "/opt/app-root/src",
        "data_path": "/nonexistent/dataset.jsonl",
        "ckpt_output_dir": "/opt/app-root/src/checkpoints",
        "data_output_dir": "/opt/app-root/src/sft-data",
        "effective_batch_size": 128,
        "num_epochs": 1,
        "max_seq_len": 4096,
        "max_batch_len": 10000,
        "learning_rate": 5e-6,
    },
    expected_error="FileNotFoundError",
    runtime=th_runtime,
    pvc_name=PVC_NAME,
)

In [ ]:
# Scenario 2: Invalid Model Path
result_2 = run_failure_scenario(
    client=client,
    scenario_name="Invalid Model Path",
    training_params={
        "model_path": "/nonexistent/model",
        "data_path": "/opt/app-root/src/data.jsonl",
        "ckpt_output_dir": "/opt/app-root/src/checkpoints",
        "data_output_dir": "/opt/app-root/src/sft-data",
        "effective_batch_size": 128,
        "num_epochs": 1,
        "max_seq_len": 4096,
        "max_batch_len": 10000,
        "learning_rate": 5e-6,
    },
    expected_error="FileNotFoundError",
    runtime=th_runtime,
    pvc_name=PVC_NAME,
)

In [ ]:
# Scenario 3: Invalid Training Hyperparameters
# Uses a string value for max_batch_len where an int is expected,
# which triggers a Pydantic ValidationError during config parsing.
result_3 = run_failure_scenario(
    client=client,
    scenario_name="Invalid Training Hyperparameters",
    training_params={
        "model_path": "/opt/app-root/src",
        "data_path": "/opt/app-root/src/data.jsonl",
        "ckpt_output_dir": "/opt/app-root/src/checkpoints",
        "data_output_dir": "/opt/app-root/src/sft-data",
        "effective_batch_size": 128,
        "num_epochs": 1,
        "max_seq_len": 4096,
        "max_batch_len": "not_a_number",
        "learning_rate": 5e-6,
    },
    expected_error="ValidationError",
    runtime=th_runtime,
    pvc_name=PVC_NAME,
)

In [ ]:
# Scenario 4: OSFT — Invalid unfreeze_rank_ratio
# The OSFT algorithm validates unfreeze_rank_ratio with a manual range check
# (0.0 <= val <= 1.0), so passing a string triggers a TypeError.
result_4 = run_failure_scenario(
    client=client,
    scenario_name="OSFT Invalid Unfreeze Rank Ratio",
    training_params={
        "model_path": "/opt/app-root/src",
        "data_path": "/opt/app-root/src/data.jsonl",
        "ckpt_output_dir": "/opt/app-root/src/checkpoints",
        "data_output_path": "/opt/app-root/src/osft-data",
        "effective_batch_size": 128,
        "num_epochs": 1,
        "max_seq_len": 2048,
        "max_tokens_per_gpu": 32000,
        "learning_rate": 5e-6,
        "unfreeze_rank_ratio": "not_a_number",  # Invalid: string instead of float
    },
    expected_error="TypeError",
    runtime=th_runtime,
    pvc_name=PVC_NAME,
    algorithm=TrainingHubAlgorithms.OSFT,
)

In [ ]:
# Scenario 5: LORA — Invalid model path
# LORA uses Unsloth's FastLanguageModel which raises a RuntimeError when
# the model path doesn't exist, before any LoRA-specific validation runs.
result_5 = run_failure_scenario(
    client=client,
    scenario_name="LORA Invalid Model Path",
    training_params={
        "model_path": "/nonexistent/model",
        "data_path": "/opt/app-root/src/data.jsonl",
        "ckpt_output_dir": "/opt/app-root/src/checkpoints",
        "data_output_dir": "/opt/app-root/src/sft-data",
        "micro_batch_size": 16,
        "num_epochs": 1,
        "max_seq_len": 1024,
        "learning_rate": 2e-4,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.0,
        "dataset_type": "chat_template",
        "field_messages": "messages",
    },
    expected_error="RuntimeError",
    runtime=th_runtime,
    pvc_name=PVC_NAME,
    algorithm=TrainingHubAlgorithms.LORA_SFT,
)

In [ ]:
# Summary
results = {
    "Invalid Dataset Path": result_1,
    "Invalid Model Path": result_2,
    "Invalid Training Hyperparameters": result_3,
    "OSFT Invalid Unfreeze Rank Ratio": result_4,
    "LORA Invalid Model Path": result_5,
}

print("\n" + "=" * 60)
print("FAILURE SCENARIOS TEST SUMMARY")
print("=" * 60)
for name, passed in results.items():
    status = "PASSED" if passed else "FAILED"
    print(f"  {name}: {status}")

all_passed = all(results.values())
print("=" * 60)
if all_passed:
    print("NOTEBOOK_STATUS: SUCCESS")
else:
    print("NOTEBOOK_STATUS: FAILURE")